In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("cleaned_patents(1).csv")

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_patents(1).csv'

# Remove the empty rows

In [ ]:
data = data.dropna(subset=['clean_text'])
data = data[data['clean_text'].astype(str).str.strip() != ""]
print(f"Rows remaining after removing empty data: {len(data)}")

Rows remaining after removing empty data: 1987


Defining the scoring function

In [ ]:
def compute_genuine_novelty_score(text):
    if pd.isna(text):
        return 0
    
    text = str(text).lower()

    breakthrough_keywords = [
        'breakthrough', 'revolutionary', 'unprecedented', 'pioneer', 
        'fundamental', 'novelty', 'disruptive', 'paradigm shift'
    ]
    
    high_keywords = [
        'novel', 'inventive', 'unique', 'innovative', 'original', 
        'state-of-the-art', 'sophisticated', 'creative'
    ]
    
    mod_keywords = [
        'new', 'improved', 'enhanced', 'advanced', 'optimized', 
        'efficient', 'modern', 'superior', 'augmented'
    ]
    
    neg_keywords = [
        'conventional', 'prior art', 'standard', 'known', 'traditional', 
        'existing', 'previous', 'typical', 'ordinary', 'common'
    ]
    
    score = (sum(text.count(kw) for kw in breakthrough_keywords) * 5.0 +
                sum(text.count(kw) for kw in high_keywords) * 3.0 +
                sum(text.count(kw) for kw in mod_keywords) * 1.5 -
                sum(text.count(kw) for kw in neg_keywords) * 1.0)
    
    words = text.split()
    if len(words) > 0:
        diversity = len(set(words)) / len(words)
        score += (len(words) / 100) * diversity
        
    return score


# Applying the scoring function

In [ ]:
data['raw_score'] = data['clean_text'].apply(compute_genuine_novelty_score)
data.columns

Index(['application_number', 'title', 'abstract', 'claims', 'text',
       'clean_text', 'raw_score'],
      dtype='object')

# Assign Tiers based on absolute thresholds

In [ ]:
def assign_tier(score):
    if score >= 15:
        return 3
    elif score >= 5:
        return 2
    elif score >= 1:
        return 1
    else:
        return 0

# Applying the tiers

In [ ]:
data['novelty tier'] = data['raw_score'].apply(assign_tier)

# Selecting only the required columns

In [ ]:
labeled_data = data[['clean_text', 'novelty tier']]

create and save a data as CSV file

In [ ]:
labeled_data.to_csv("labeled_patents.csv", index=False)
print(f"File saved successfully as {'labeled_patents.csv'}")
print("Tier Distribution:")
print(labeled_data['novelty tier'].value_counts().sort_index())

File saved successfully as labeled_patents.csv
Tier Distribution:
novelty tier
0     570
1    1258
2     107
3      52
Name: count, dtype: int64


# _______________________________________________________

# Balancing the DataSet

In [ ]:
unbalenced_data= pd.read_csv("labeled_patents.csv")

In [ ]:
balanced_data = unbalenced_data.groupby('novelty tier').apply(
    lambda x: x.sample(n=50, random_state=42)
).reset_index(drop=True)


C:\Users\prade\AppData\Local\Temp\ipykernel_31164\2251446785.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_data = unbalenced_data.groupby('novelty tier').apply(


# generating balenced_data.csv

In [ ]:
balanced_data.to_csv('balanced_data.csv', index=False)

print("\nDistribution after balancing:")
print(balanced_data['novelty tier'].value_counts().sort_index())
print(f"\nBalanced data successfully saved to {'balanced_data.csv'}")


Distribution after balancing:
novelty tier
0    50
1    50
2    50
3    50
Name: count, dtype: int64

Balanced data successfully saved to balanced_data.csv


# ___________________________________________________________

# Shuffle the Balanced dataset

In [ ]:
balanced_dataset = pd.read_csv('balanced_data.csv')
balanced_dataset_shuffled = balanced_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
balanced_dataset_shuffled.to_csv('final_dataset_balanced.csv', index=False)
print(f"Shuffled data saved successfully to final_dataset_balanced.csv")

Shuffled data saved successfully to final_dataset_balanced.csv
